# Pass 2 & 3 — Batch API Versions

Same workflow as `Pass_1_Batch.ipynb`: submit → close laptop → retrieve results.  
Both passes use the Message Batches API (50% cheaper, no rate-limit management).

**Workflow:**
1. Run **Part A** (Pass 2): submit consolidation batch, wait, download, save  
2. Run **Part B** (Pass 3): submit verification batch, wait, download, save  

Each part is self-contained — you can restart the kernel between them.

In [2]:
# === Shared: Imports & Config ===

import pandas as pd
import time, os, json, re
import anthropic
from pathlib import Path
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])
MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic()

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective', 'KIID Objective/Investment Policy',
    'Prospectus Objective', 'Investment Strategy - English',
    'PRIIPS KID Objective - Danish', 'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish', 'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German', 'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian', 'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish', 'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish', 'Investment Strategy - Finnish',
    'Investment Strategy - French', 'Investment Strategy - German',
    'Investment Strategy - Italian', 'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese', 'Investment Strategy - Spanish',
    'Investment Strategy - Swedish', 'Strategy Description'
]


def robust_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    for char, esc in [('\u201E','\\u201E'),('\u201C','\\u201C'),('\u201D','\\u201D'),
                      ('\u00AB','\\u00AB'),('\u00BB','\\u00BB'),('\u201A','\\u201A'),
                      ('\u2018','\\u2018'),('\u2019','\\u2019')]:
        cleaned = cleaned.replace(char, esc)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    def fix_strings(match):
        s = match.group(0)
        s = s.replace('\n', '\\n').replace('\r', '\\r').replace('\t', '\\t')
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass
    brace_match = re.search(r'\{.*\}', fixed, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass
    return None


def check_batch_status(batch_id):
    """Check and print batch status. Returns True if ended."""
    status = client.messages.batches.retrieve(batch_id)
    rc = status.request_counts
    total = rc.processing + rc.succeeded + rc.errored + rc.canceled + rc.expired
    done = total - rc.processing
    print(f"Batch: {batch_id}")
    print(f"Status: {status.processing_status}")
    print(f"  Succeeded: {rc.succeeded} | Errored: {rc.errored} | Expired: {rc.expired} | Processing: {rc.processing}")
    if total > 0:
        print(f"  Progress: {done}/{total} ({done/total*100:.1f}%)")
    if status.processing_status == 'ended':
        print("  COMPLETE — ready to download.")
        return True
    return False

---
# Part A — Pass 2: Consolidate & Deduplicate
---

In [3]:
# === A1: Pass 2 System Prompt & Few-Shot (identical to your notebook) ===

PASS2_SYSTEM_PROMPT = """You are consolidating fund objective extractions that were made independently from multiple regulatory text columns for the same European mutual fund.

You will receive a JSON object where each key is a column name, and the value contains:
- "language": the language of that column
- "objectives": a list of objectives extracted from that column, each with:
  - "objective_text": verbatim text from the source
  - "objective_text_english": English translation
  - "objective_type": "financial" or "sustainable"

YOUR TASK:
1. MATCH equivalent objectives across columns/languages. The same objective may appear in English, French, German, Swedish, etc. These are duplicates and should be consolidated into ONE entry.
2. DEDUPLICATE: If multiple columns express the same objective (even in different words or languages), keep it only once.
3. For each unique objective, select the BEST English phrasing — prefer a native English source if available; otherwise use or improve the translation.
4. Classify each as "financial" or "sustainable".
5. Record which columns contained this objective (for traceability).

MATCHING GUIDANCE:
- "long-term capital growth" in English and "croissance du capital à long terme" in French = SAME objective
- "outperform the benchmark" and "exceed the benchmark index" = SAME objective (minor wording variation)
- "achieve capital growth" and "achieve capital growth and outperform the benchmark" — the second contains TWO objectives; match the first part and keep the second as separate
- Be generous in matching across languages but strict about not merging genuinely different objectives

OUTPUT FORMAT:
{
  "consolidated_objectives": [
    {
      "objective_number": 1,
      "objective_text_english": "the final English text for this objective",
      "objective_type": "financial" or "sustainable",
      "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", ...],
      "match_notes": "brief note on how columns were matched, or null if only found in one column"
    }
  ],
  "consolidation_notes": "any important notes about the consolidation process"
}

If Pass 1 found NO objectives in ANY column:
{
  "consolidated_objectives": [],
  "consolidation_notes": "NOT IDENTIFIED — no objectives found in any column"
}
"""

PASS2_FEW_SHOT = [
    {
        "fund_name": "Example Multilingual Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {"objective_text": "achieve capital growth", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "objective_text_english": "outperform the benchmark", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {"objective_text": "réaliser une croissance du capital", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "surperformer l'indice de référence", "objective_text_english": "outperform the benchmark index", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - German": {
                "language": "German",
                "objectives": [
                    {"objective_text": "Kapitalwachstum erzielen", "objective_text_english": "achieve capital growth", "objective_type": "financial"},
                    {"objective_text": "die Benchmark übertreffen", "objective_text_english": "outperform the benchmark", "objective_type": "financial"},
                    {"objective_text": "Reduzierung der Treibhausgasemissionen", "objective_text_english": "reduction of greenhouse gas emissions", "objective_type": "sustainable"}
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {"objective_number": 1, "objective_text_english": "achieve capital growth", "objective_type": "financial", "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"], "match_notes": "Same objective across English, French, and German columns"},
                {"objective_number": 2, "objective_text_english": "outperform the benchmark", "objective_type": "financial", "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"], "match_notes": "Same benchmark-beating objective across all three languages"},
                {"objective_number": 3, "objective_text_english": "reduction of greenhouse gas emissions", "objective_type": "sustainable", "found_in_columns": ["PRIIPS KID Objective - German"], "match_notes": "Sustainability objective found only in German column"}
            ],
            "consolidation_notes": "Two financial objectives matched across all three languages. One sustainability objective found only in the German column."
        }
    }
]

In [4]:
# === A2: Load Pass 1 Output & Build Pass 2 Batch ===
# UPDATE this path to your actual Pass 1 output file
PASS1_FILE = os.path.join(OUTPUT_DIR, "Pass1_Extract_5667_funds_20260528_1706.xlsx")  # UPDATE

p1_df = pd.read_excel(PASS1_FILE)
p1_df['pass1_raw'] = p1_df['pass1_raw'].apply(json.loads)
p1_df['columns_sent'] = p1_df['columns_sent'].apply(json.loads)
print(f"Loaded {len(p1_df)} funds from Pass 1")

# Build batch requests
p2_batch_requests = []
p2_metadata = {}

for idx in range(len(p1_df)):
    row = p1_df.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Fund_Name']
    pass1_data = row['pass1_raw']

    # Skip Pass 1 errors
    if '_error' in pass1_data:
        continue

    # Filter to columns with objectives
    cols_with_data = {}
    for col, data in pass1_data.items():
        if col.startswith('_'):
            continue
        if isinstance(data, dict) and 'objectives' in data and len(data['objectives']) > 0:
            cols_with_data[col] = data

    if not cols_with_data:
        continue

    custom_id = re.sub(r'[^a-zA-Z0-9_-]', '_', f"p2-{fund_id}")[:64]

    # Build messages (few-shot + real request)
    messages = []
    for ex in PASS2_FEW_SHOT:
        messages.append({
            "role": "user",
            "content": f"Fund Name: {ex['fund_name']}\n\nPass 1 extractions (per-column):\n{json.dumps(ex['pass1_data'], indent=2)}"
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(ex["response"], indent=2)
        })
    messages.append({
        "role": "user",
        "content": f"Fund ID: {fund_id}\nFund Name: {fund_name}\n\nPass 1 extractions (per-column):\n{json.dumps(cols_with_data, indent=2)}"
    })

    p2_batch_requests.append(
        Request(
            custom_id=custom_id,
            params=MessageCreateParamsNonStreaming(
                model=MODEL, max_tokens=2000, temperature=0,
                system=PASS2_SYSTEM_PROMPT, messages=messages
            )
        )
    )
    p2_metadata[custom_id] = {'FundId': fund_id, 'Fund_Name': fund_name}

print(f"Pass 2 batch: {len(p2_batch_requests)} requests ready")

# Save metadata
with open(os.path.join(OUTPUT_DIR, 'p2_batch_metadata.json'), 'w') as f:
    json.dump(p2_metadata, f)

Loaded 5667 funds from Pass 1
Pass 2 batch: 5540 requests ready


In [5]:
# === A3: Submit Pass 2 Batch ===

print(f"Submitting Pass 2 batch ({len(p2_batch_requests)} requests)...")
p2_batch = client.messages.batches.create(requests=p2_batch_requests)

P2_BATCH_ID = p2_batch.id
with open(os.path.join(OUTPUT_DIR, 'p2_batch_id.txt'), 'w') as f:
    f.write(P2_BATCH_ID)

print(f"Batch ID: {P2_BATCH_ID}")
print(f"Status: {p2_batch.processing_status}")
print(f"Saved to p2_batch_id.txt — you can close the notebook now.")

Submitting Pass 2 batch (5540 requests)...
Batch ID: msgbatch_01FtjABz6MewYtfsRt9wZ8ZN
Status: in_progress
Saved to p2_batch_id.txt — you can close the notebook now.


In [7]:
# === A4: Check Pass 2 Status (run anytime) ===

with open(os.path.join(OUTPUT_DIR, 'p2_batch_id.txt'), 'r') as f:
    P2_BATCH_ID = f.read().strip()
check_batch_status(P2_BATCH_ID)

Batch: msgbatch_01FtjABz6MewYtfsRt9wZ8ZN
Status: ended
  Succeeded: 5540 | Errored: 0 | Expired: 0 | Processing: 0
  Progress: 5540/5540 (100.0%)
  COMPLETE — ready to download.


True

In [9]:
# === A5: Download & Parse Pass 2 Results ===

with open(os.path.join(OUTPUT_DIR, 'p2_batch_id.txt'), 'r') as f:
    P2_BATCH_ID = f.read().strip()
with open(os.path.join(OUTPUT_DIR, 'p2_batch_metadata.json'), 'r') as f:
    p2_metadata = json.load(f)

# Also need p1_df to capture Pass-1-error funds
PASS1_FILE = os.path.join(OUTPUT_DIR, "Pass1_Extract_5667_funds_20260528_1706.xlsx")  # UPDATE (same as A2)
p1_df = pd.read_excel(PASS1_FILE)
p1_df['pass1_raw'] = p1_df['pass1_raw'].apply(json.loads)

p2_results = []
succeeded = errored = 0

for result in client.messages.batches.results(P2_BATCH_ID):
    meta = p2_metadata.get(result.custom_id, {'FundId': result.custom_id, 'Fund_Name': 'UNKNOWN'})
    if result.result.type == 'succeeded':
        succeeded += 1
        text = result.result.message.content[0].text if result.result.message.content else ''
        parsed = robust_json_parse(text)
        p2_results.append({'FundId': meta['FundId'], 'Fund_Name': meta['Fund_Name'],
                           'pass2_raw': parsed if parsed else {'_error': f'JSON parse fail: {text[:300]}'}})
    else:
        errored += 1
        p2_results.append({'FundId': meta['FundId'], 'Fund_Name': meta['Fund_Name'],
                           'pass2_raw': {'_error': f'{result.result.type}: {getattr(result.result, "error", "unknown")}'}})

# Add back funds that were skipped (Pass 1 errors or no objectives)
downloaded_ids = {r['FundId'] for r in p2_results}
for _, row in p1_df.iterrows():
    if row['FundId'] not in downloaded_ids:
        raw = row['pass1_raw']
        reason = raw.get('_error', 'No objectives found in any column') if '_error' in raw else 'No objectives found in any column'
        p2_results.append({'FundId': row['FundId'], 'Fund_Name': row['Fund_Name'],
                           'pass2_raw': {'consolidated_objectives': [], 'consolidation_notes': f'Skipped — {reason}'}})

pass2_df = pd.DataFrame(p2_results)
print(f"Downloaded: {succeeded} succeeded, {errored} errored")
print(f"Total funds (incl skipped): {len(pass2_df)}")

Downloaded: 5540 succeeded, 0 errored
Total funds (incl skipped): 5667


In [10]:
# === A6: Flatten & Summary Stats ===

flat_rows = []
for _, row in pass2_df.iterrows():
    raw = row['pass2_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}
    if '_error' in raw:
        base.update({'Number_of_Objectives': 0, 'Number_Financial': 0, 'Number_Sustainable': 0, 'Consolidation_Notes': raw['_error']})
        flat_rows.append(base); continue
    objs = raw.get('consolidated_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Number_Financial'] = sum(1 for o in objs if o.get('objective_type') == 'financial')
    base['Number_Sustainable'] = sum(1 for o in objs if o.get('objective_type') == 'sustainable')
    base['Consolidation_Notes'] = raw.get('consolidation_notes', '')
    for i in range(5):
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}'] = o.get('objective_text_english', '')
            base[f'Objective_{i+1}_Type'] = o.get('objective_type', '')
            base[f'Objective_{i+1}_Columns'] = ', '.join(o.get('found_in_columns', []))
        else:
            base[f'Objective_{i+1}'] = None; base[f'Objective_{i+1}_Type'] = None; base[f'Objective_{i+1}_Columns'] = None
    flat_rows.append(base)

pass2_flat = pd.DataFrame(flat_rows)
total_obj = pass2_flat['Number_of_Objectives'].sum()
print("PASS 2 SUMMARY:")
print(f"  Funds: {len(pass2_flat)}")
print(f"  With ≥1 objective: {(pass2_flat['Number_of_Objectives'] > 0).sum()}")
print(f"  Avg objectives: {pass2_flat['Number_of_Objectives'].mean():.1f}")
print(f"  Financial: {pass2_flat['Number_Financial'].sum()}, Sustainable: {pass2_flat['Number_Sustainable'].sum()}")
print(f"  Funds with ≥1 sustainable: {(pass2_flat['Number_Sustainable'] > 0).sum()}")

PASS 2 SUMMARY:
  Funds: 5667
  With ≥1 objective: 5537
  Avg objectives: 1.9
  Financial: 8341, Sustainable: 2700
  Funds with ≥1 sustainable: 1952


In [11]:
# === A7: Save Pass 2 Output ===

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Flattened (human-readable)
pass2_flat.to_excel(os.path.join(OUTPUT_DIR, f'Pass2_Consolidated_{len(pass2_flat)}_funds_{timestamp}.xlsx'), index=False, engine='openpyxl')

# Raw JSON (for Pass 3 input)
p2_raw_df = pass2_df.copy()
p2_raw_df['pass2_raw'] = p2_raw_df['pass2_raw'].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x)
p2_raw_path = os.path.join(OUTPUT_DIR, f'Pass2_Raw_{len(pass2_df)}_funds_{timestamp}.xlsx')
p2_raw_df.to_excel(p2_raw_path, index=False, engine='openpyxl')

print(f"Saved flattened: Pass2_Consolidated_{len(pass2_flat)}_funds_{timestamp}.xlsx")
print(f"Saved raw JSON:  Pass2_Raw_{len(pass2_df)}_funds_{timestamp}.xlsx")
print(f"  → Use the raw JSON file as input to Pass 3 (Part B below)")

Saved flattened: Pass2_Consolidated_5667_funds_20260528_1847.xlsx
Saved raw JSON:  Pass2_Raw_5667_funds_20260528_1847.xlsx
  → Use the raw JSON file as input to Pass 3 (Part B below)


---
# Part B — Pass 3: Verify Against Source Text
---

In [12]:
# === B1: Pass 3 System Prompt (identical to your notebook) ===

PASS3_SYSTEM_PROMPT = """You are verifying extracted fund objectives against the original regulatory source text.

You will receive:
1. A list of consolidated objectives (in English) from Pass 2
2. The original source text from ALL available columns (in various languages)

YOUR TASK:
For EACH objective, determine whether it can be traced back to text in ANY of the source columns.

VERIFICATION RULES:
- An objective is VERIFIED if you can find corresponding text in at least one source column.
  The match can be in any language — the objective may be an English translation of French/German/etc. source text.
- An objective is FLAGGED if you cannot find any corresponding text in any column.
  This means it may have been hallucinated or incorrectly inferred.

CLASSIFICATION CHECK:
- Also verify whether each objective is correctly classified as "financial" or "sustainable".
- If the classification is wrong, provide the correct one.

OUTPUT FORMAT:
{
  "verified_objectives": [
    {
      "objective_number": 1,
      "objective_text_english": "the objective text from Pass 2",
      "verification_status": "VERIFIED" or "FLAGGED",
      "verified_in_column": "column name where found" or null,
      "source_quote": "brief quote or paraphrase from source showing the match" or null,
      "objective_type": "financial" or "sustainable",
      "type_changed": false,
      "verification_notes": "brief explanation"
    }
  ],
  "overall_confidence": "high" or "medium" or "low",
  "verification_summary": "brief summary of verification results"
}
"""

In [13]:
# === B2: Load Pass 2 Output + Source Data & Build Pass 3 Batch ===
# UPDATE this path to your actual Pass 2 raw output file
PASS2_RAW_FILE = os.path.join(OUTPUT_DIR, "Pass2_Raw_5667_funds_20260528_1847.xlsx")  # UPDATE

p2_df = pd.read_excel(PASS2_RAW_FILE)
p2_df['pass2_raw'] = p2_df['pass2_raw'].apply(json.loads)
print(f"Loaded {len(p2_df)} funds from Pass 2")

# Load original source data for verification
df_source = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df_source)} funds from source data")

# Build batch
p3_batch_requests = []
p3_metadata = {}

for idx in range(len(p2_df)):
    row = p2_df.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Fund_Name']
    pass2_data = row['pass2_raw']

    # Skip errors and empty
    if '_error' in pass2_data:
        continue
    objs = pass2_data.get('consolidated_objectives', [])
    if not objs:
        continue

    # Get source columns
    fund_source = df_source[df_source['FundId'] == fund_id]
    if fund_source.empty:
        continue
    srow = fund_source.iloc[0]
    source_cols = {}
    for col in OBJECTIVE_COLUMNS:
        if col in srow.index:
            val = srow[col]
            if pd.notna(val) and str(val).strip() not in ['-', 'Not available', '']:
                source_cols[col] = str(val)

    if not source_cols:
        continue

    custom_id = re.sub(r'[^a-zA-Z0-9_-]', '_', f"p3-{fund_id}")[:64]

    source_text = "\n\n".join([f"=== Column: {c} ===\n{v}" for c, v in source_cols.items()])
    objectives_text = json.dumps(objs, indent=2)

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

CONSOLIDATED OBJECTIVES FROM PASS 2:
{objectives_text}

ORIGINAL SOURCE TEXT (all available columns):
{source_text}"""

    p3_batch_requests.append(
        Request(
            custom_id=custom_id,
            params=MessageCreateParamsNonStreaming(
                model=MODEL, max_tokens=2000, temperature=0,
                system=PASS3_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}]
            )
        )
    )
    p3_metadata[custom_id] = {'FundId': fund_id, 'Fund_Name': fund_name}

print(f"Pass 3 batch: {len(p3_batch_requests)} requests ready")

with open(os.path.join(OUTPUT_DIR, 'p3_batch_metadata.json'), 'w') as f:
    json.dump(p3_metadata, f)

Loaded 5667 funds from Pass 2
Loaded 5680 funds from source data
Pass 3 batch: 5537 requests ready


In [14]:
# === B3: Submit Pass 3 Batch ===

print(f"Submitting Pass 3 batch ({len(p3_batch_requests)} requests)...")
p3_batch = client.messages.batches.create(requests=p3_batch_requests)

P3_BATCH_ID = p3_batch.id
with open(os.path.join(OUTPUT_DIR, 'p3_batch_id.txt'), 'w') as f:
    f.write(P3_BATCH_ID)

print(f"Batch ID: {P3_BATCH_ID}")
print(f"Saved to p3_batch_id.txt — you can close the notebook now.")

Submitting Pass 3 batch (5537 requests)...
Batch ID: msgbatch_01ESeouJqD5ZLJWj18Jef1Y6
Saved to p3_batch_id.txt — you can close the notebook now.


In [16]:
# === B4: Check Pass 3 Status (run anytime) ===

with open(os.path.join(OUTPUT_DIR, 'p3_batch_id.txt'), 'r') as f:
    P3_BATCH_ID = f.read().strip()
check_batch_status(P3_BATCH_ID)


Batch: msgbatch_01ESeouJqD5ZLJWj18Jef1Y6
Status: ended
  Succeeded: 5483 | Errored: 54 | Expired: 0 | Processing: 0
  Progress: 5537/5537 (100.0%)
  COMPLETE — ready to download.


True

In [20]:
# === B5: Download & Parse Pass 3 Results ===

with open(os.path.join(OUTPUT_DIR, 'p3_batch_id.txt'), 'r') as f:
    P3_BATCH_ID = f.read().strip()
with open(os.path.join(OUTPUT_DIR, 'p3_batch_metadata.json'), 'r') as f:
    p3_metadata = json.load(f)

# Reload Pass 2 data for skipped funds
PASS2_RAW_FILE = os.path.join(OUTPUT_DIR, "Pass2_Raw_5667_funds_20260528_1847.xlsx")  # UPDATE (same as B2)
p2_df = pd.read_excel(PASS2_RAW_FILE)
p2_df['pass2_raw'] = p2_df['pass2_raw'].apply(json.loads)

p3_results = []
succeeded = errored = 0

for result in client.messages.batches.results(P3_BATCH_ID):
    meta = p3_metadata.get(result.custom_id, {'FundId': result.custom_id, 'Fund_Name': 'UNKNOWN'})
    if result.result.type == 'succeeded':
        succeeded += 1
        text = result.result.message.content[0].text if result.result.message.content else ''
        parsed = robust_json_parse(text)
        p3_results.append({'FundId': meta['FundId'], 'Fund_Name': meta['Fund_Name'],
                           'pass3_raw': parsed if parsed else {'_error': f'JSON parse fail: {text[:300]}'}})
    else:
        errored += 1
        p3_results.append({'FundId': meta['FundId'], 'Fund_Name': meta['Fund_Name'],
                           'pass3_raw': {'_error': f'{result.result.type}: {getattr(result.result, "error", "unknown")}'}})

# Add back skipped funds
downloaded_ids = {r['FundId'] for r in p3_results}
for _, row in p2_df.iterrows():
    if row['FundId'] not in downloaded_ids:
        p3_results.append({'FundId': row['FundId'], 'Fund_Name': row['Fund_Name'],
                           'pass3_raw': {'verified_objectives': [], 'overall_confidence': 'none',
                                         'verification_summary': 'Skipped — no objectives to verify'}})

pass3_df = pd.DataFrame(p3_results)
print(f"Downloaded: {succeeded} succeeded, {errored} errored")
print(f"Total funds (incl skipped): {len(pass3_df)}")

Downloaded: 5483 succeeded, 54 errored
Total funds (incl skipped): 5667


In [21]:
# === B6: Flatten into Final Output + Summary ===

from collections import Counter

final_rows = []
for _, row in pass3_df.iterrows():
    raw = row['pass3_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}
    if '_error' in raw:
        base.update({'Number_of_Objectives': 0, 'Overall_Confidence': 'error',
                      'Verification_Summary': raw['_error'], 'Has_Flagged': False})
        final_rows.append(base); continue
    objs = raw.get('verified_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Overall_Confidence'] = raw.get('overall_confidence', '')
    base['Verification_Summary'] = raw.get('verification_summary', '')
    base['Has_Flagged'] = any(o.get('verification_status') == 'FLAGGED' for o in objs)
    for i in range(5):
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}'] = o.get('objective_text_english', '')
            base[f'Objective_{i+1}_Type'] = o.get('objective_type', '')
            base[f'Objective_{i+1}_Status'] = o.get('verification_status', '')
            base[f'Objective_{i+1}_Verified_In'] = o.get('verified_in_column', '')
            base[f'Objective_{i+1}_Type_Changed'] = o.get('type_changed', False)
            base[f'Objective_{i+1}_Notes'] = o.get('verification_notes', '')
        else:
            for suffix in ['', '_Type', '_Status', '_Verified_In', '_Type_Changed', '_Notes']:
                base[f'Objective_{i+1}{suffix}'] = None
    final_rows.append(base)

final_df = pd.DataFrame(final_rows)

total = len(final_df)
with_obj = (final_df['Number_of_Objectives'] > 0).sum()
flagged_funds = final_df['Has_Flagged'].sum()
print("=" * 80)
print("FINAL VERIFICATION SUMMARY")
print("=" * 80)
print(f"  Funds processed: {total}")
print(f"  Funds with objectives: {with_obj} ({with_obj/total*100:.1f}%)")
print(f"  Funds with FLAGGED objectives: {flagged_funds} ({flagged_funds/total*100:.1f}%)")

all_statuses = []
for col in [f'Objective_{i}_Status' for i in range(1, 6)]:
    all_statuses.extend(final_df[col].dropna().tolist())
if all_statuses:
    print(f"\n  Objective-level verification:")
    for s, c in Counter(all_statuses).items():
        print(f"    {s}: {c} ({c/len(all_statuses)*100:.1f}%)")

type_changes = sum(1 for col in [f'Objective_{i}_Type_Changed' for i in range(1, 6)]
                   for v in final_df[col].dropna() if v == True)
print(f"\n  Classification changes: {type_changes}")
print(f"\nConfidence distribution:")
print(final_df['Overall_Confidence'].value_counts())

FINAL VERIFICATION SUMMARY
  Funds processed: 5667
  Funds with objectives: 5368 (94.7%)
  Funds with FLAGGED objectives: 32 (0.6%)

  Objective-level verification:
    VERIFIED: 10522 (99.7%)
    FLAGGED: 34 (0.3%)

  Classification changes: 57

Confidence distribution:
Overall_Confidence
high      5298
error      169
none       130
medium      65
low          5
Name: count, dtype: int64


In [22]:
# === B7: Show Flagged Objectives ===

flagged_df = final_df[final_df['Has_Flagged'] == True]
if len(flagged_df) > 0:
    print(f"FLAGGED FOR MANUAL REVIEW: {len(flagged_df)} funds")
    for _, row in flagged_df.iterrows():
        print(f"\n  Fund: {row['Fund_Name']} ({row['FundId']})")
        for i in range(1, 6):
            if row.get(f'Objective_{i}_Status') == 'FLAGGED':
                print(f"    FLAGGED Objective {i}: {row[f'Objective_{i}']}")
                print(f"      Notes: {row[f'Objective_{i}_Notes']}")
else:
    print("No flagged objectives — all verified successfully.")

FLAGGED FOR MANUAL REVIEW: 32 funds

  Fund: Mirova Women Ldrs & Divst Eq I/A EUR (FS0000FA24)
    FLAGGED Objective 1: maximise the return on your investment through a combination of capital growth and income on the Fund's assets
      Notes: This objective cannot be found in any of the source columns. The PRIIPS KID Objective column (the only column claimed to contain it) does not include any language about 'maximise the return on your investment through a combination of capital growth and income'. The PRIIPS KID Objective instead describes a sustainable investment objective focused on allocating capital to sustainable economic models. This objective appears to have been hallucinated or incorrectly attributed.

  Fund: Lannebo Småbolag A (FSGBR053AV)
    FLAGGED Objective 3: The objective is to achieve the highest possible value through active allocation of the fund's assets between equity and fixed income investments, while achieving good risk diversification.
      Notes: FLAGGED d

In [23]:
# === B8: Save Final Output ===

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

final_filename = f'FINAL_Verified_{len(final_df)}_funds_{timestamp}.xlsx'
final_df.to_excel(os.path.join(OUTPUT_DIR, final_filename), index=False, engine='openpyxl')

if len(flagged_df) > 0:
    flagged_filename = f'FLAGGED_ManualReview_{len(flagged_df)}_funds_{timestamp}.xlsx'
    flagged_df.to_excel(os.path.join(OUTPUT_DIR, flagged_filename), index=False, engine='openpyxl')
    print(f"Saved flagged:  {flagged_filename}")

print(f"Saved final:    {final_filename}")
print(f"\nDone. Three-pass extraction complete.")

Saved flagged:  FLAGGED_ManualReview_32_funds_20260529_1240.xlsx
Saved final:    FINAL_Verified_5667_funds_20260529_1240.xlsx

Done. Three-pass extraction complete.


In [24]:
# Non-high OR has any flagged objective
review = final_df[(final_df['Overall_Confidence'] != 'high') | (final_df['Has_Flagged'] == True)].copy()
print(f"Total for review: {len(review)}")

Total for review: 379


In [25]:
review.to_excel(os.path.join(OUTPUT_DIR, f'REVIEW_non_high_confidence_{len(review)}_funds.xlsx'), index=False, engine='openpyxl')


In [26]:
client = anthropic.Anthropic()

# Add your batch IDs for each pass
batch_ids = {
    'Pass 1': 'msgbatch_011kkZt8nUokSNAqTtrvv2XV',
    'Pass 2': 'msgbatch_01FtjABz6MewYtfsRt9wZ8ZN', 
    'Pass 3': 'msgbatch_01ESeouJqD5ZLJWj18Jef1Y6'
}

total_cost = 0

for pass_name, batch_id in batch_ids.items():
    input_tokens, output_tokens = 0, 0
    
    for result in client.messages.batches.results(batch_id):
        if result.result.type == 'succeeded':
            input_tokens  += result.result.message.usage.input_tokens
            output_tokens += result.result.message.usage.output_tokens
    
    # Batch API: $1.50/M input, $7.50/M output
    cost = (input_tokens / 1_000_000 * 1.50) + (output_tokens / 1_000_000 * 7.50)
    total_cost += cost
    
    print(f"{pass_name}: {input_tokens:,} input / {output_tokens:,} output → ${cost:.2f}")

print(f"\nTotal: ${total_cost:.2f}")

Pass 1: 46,931,934 input / 7,151,809 output → $124.04
Pass 2: 15,559,244 input / 2,674,831 output → $43.40
Pass 3: 33,786,694 input / 3,413,339 output → $76.28

Total: $243.72
